# Processing Predicates

This notebook demonstrates how to load positive and negative predicates from pickle files, build a domain, and initialize a `LogicModel`.

## Strategy: Sparse Matrices & Frequency-Based Probabilities

To handle a large number of predicates efficiently, we employ sparse vector representations. This allows us to scale beyond the dense core limitations.

Furthermore, we demonstrate **Distributional Semantics** by using the **frequency** of assertions in the corpus to derive confidence scores. The hypothesis is: "The more often a fact is stated, the more confident we are in its truth."

We will:
1.  Count how many times each specific fact (e.g., `(Sanders, is_candidate, None)`) appears in the raw data.
2.  Normalize these counts against the most frequent fact for that subject.
3.  Use these normalized values as probabilities (0.0 to 1.0) in our LogicModel.

**Note:** We will initialize the LogicModel with `use_sparse=True`.

In [1]:
import pickle
import LogicModel as m
import numpy as np
import collections
import scipy.sparse as sp

## 1. Load Data

We will load a larger set of predicates, as sparse matrices can handle them.

In [2]:
# Load ALL data
# Using the larger file ALL-predicates-31994.pickle
try:
    with open('Predicates/ALL-predicates-31994.pickle', 'rb') as f:
        raw_pos = pickle.load(f)
    print(f"Loaded {len(raw_pos)} positive predicates.")
except FileNotFoundError:
    print("Full pickle file not found, falling back to filtered.")
    with open('Predicates/FILTERED-predicates.pickle', 'rb') as f:
        raw_pos = pickle.load(f)
        print(f"Loaded {len(raw_pos)} filtered positive predicates.")

# We will skip negative predicates for this large scale demo to save time, or load subset
raw_neg = []


Loaded 31994 positive predicates.


## 2. Process Data for LogicModel

We filter pronouns but attempt to keep a much larger domain than before.

In [3]:
pronouns = {'he', 'she', 'it', 'him', 'her', 'they', 'them', 'we', 'us', 'you',
            'He', 'She', 'It', 'Him', 'Her', 'They', 'Them', 'We', 'Us', 'You',
            'this', 'This', 'that', 'That', 'these', 'These', 'those', 'Those', 
            'I', 'me', 'Me', 'my', 'My', 'myself', 'Myself'}

def filter_pronouns(pred_list):
    cleaned = []
    for s, p, o in pred_list:
        if s in pronouns: continue
        if o is not None and o in pronouns: continue
        cleaned.append((s, p, o))
    return cleaned

clean_pos = filter_pronouns(raw_pos)

# We will define a larger limit for elements, or even try all.
# Let's take top 2000 frequent elements to show scaling.
element_counts = collections.Counter()
for s, p, o in clean_pos:
    element_counts[s] += 1
    if o: element_counts[o] += 1

TOP_N = 2000
top_elements = set([e for e, c in element_counts.most_common(TOP_N)])
top_elements.add("Sanders") # Ensure Sanders is there

final_preds = []
for s, p, o in clean_pos:
    if s in top_elements:
        if o is None or o in top_elements:
            final_preds.append((s, p, o))

# Unique
def safe_sort_key(t):
    return tuple((x if x is not None else "") for x in t)
final_preds_unique = sorted(list(set(final_preds)), key=safe_sort_key)

# Build Dictionaries
domain_set = set()
unary_preds_dict = {}
binary_preds_dict = {}

def add_to_domain(elem):
    if elem is not None:
        domain_set.add(elem)

for s, p, o in final_preds_unique:
    add_to_domain(s)
    add_to_domain(o)
    
    if o is None:
        if p not in unary_preds_dict: unary_preds_dict[p] = []
        unary_preds_dict[p].append(s)
    else:
        if p not in binary_preds_dict: binary_preds_dict[p] = []
        binary_preds_dict[p].append((s, o))

domain_list = sorted(list(domain_set))
print(f"Domain Size: {len(domain_list)}")
print(f"Number of Unary Predicates: {len(unary_preds_dict)}")
print(f"Number of Binary Predicates: {len(binary_preds_dict)}")

Domain Size: 1994
Number of Unary Predicates: 2420
Number of Binary Predicates: 1006


## 3. Build Logic Model (Sparse)

Here we enable `use_sparse=True`.

In [4]:
model = m.LogicModel(
    listOfElements=domain_list,
    dictionaryOfUnaryPredicates=unary_preds_dict,
    dictionaryOfBinaryPredicates=binary_preds_dict,
    use_sparse=True
)

print("Building sparse model...")
model.buildAll()
print("Model built!")

Building sparse model...


Model built!


## 4. Deriving Probabilities from Corpus Frequency

We perform the same analysis for **"Sanders"**.

In [5]:
target = "Sanders"

# 1. Count Predicate Occurrences for Target in our filtered set
predicate_counts = collections.Counter()

for s, p, o in final_preds:
    if s == target:
        key = (p, o)
        predicate_counts[key] += 1

# 2. Find Max Frequency
if predicate_counts:
    max_count = predicate_counts.most_common(1)[0][1]
    print(f"\nMax Count for any single fact: {max_count}\n")
    
    # 3. Derive Probabilities & Update Model
    # Since we are using sparse model, this checks update efficiency too.
    for (pred, obj), count in predicate_counts.items():
        prob = count / max_count
        print(f"Fact: {pred}({target}, {obj}) | Count: {count} | Derived Prob: {prob:.2f}")
        
        if obj is None:
            model.updateUnaryPredicate(target, pred, prob)
        else:
            model.updateBinaryPredicate((target, obj), pred, prob)
else:
    print(f"No facts found for {target}")


Max Count for any single fact: 2

Fact: calls(Sanders, None) | Count: 1 | Derived Prob: 0.50
Fact: is_member(Sanders, None) | Count: 2 | Derived Prob: 1.00
Fact: is_candidate(Sanders, None) | Count: 1 | Derived Prob: 0.50
Fact: want(Sanders, money) | Count: 1 | Derived Prob: 0.50
Fact: studied(Sanders, None) | Count: 1 | Derived Prob: 0.50
Fact: went(Sanders, None) | Count: 1 | Derived Prob: 0.50
Fact: led(Sanders, University) | Count: 1 | Derived Prob: 0.50
Fact: began(Sanders, career) | Count: 1 | Derived Prob: 0.50
Fact: lost(Sanders, None) | Count: 1 | Derived Prob: 0.50
Fact: resigned(Sanders, None) | Count: 1 | Derived Prob: 0.50
Fact: ran(Sanders, None) | Count: 2 | Derived Prob: 1.00
Fact: left(Sanders, office) | Count: 1 | Derived Prob: 0.50
Fact: taught(Sanders, science) | Count: 1 | Derived Prob: 0.50
Fact: ran(Sanders, election) | Count: 1 | Derived Prob: 0.50
Fact: ran(Sanders, Smith) | Count: 1 | Derived Prob: 0.50
Fact: is_independent(Sanders, None) | Count: 1 | Derived

## 5. Queries (Sparse)

We run the same queries. The output should be consistent with the dense model, but calculation is done via sparse matrices.

In [6]:
is_candidate = model.unaryOp("is_candidate", target)
print(f"1. Is '{target}' a 'candidate'?\n{is_candidate.flatten()}\n")

is_senator = model.unaryOp("is_senator", target)
print(f"2. Is '{target}' a 'senator'?\n{is_senator.flatten()}\n")

candidate_and_senator = model.andOp(is_candidate, is_senator)
print(f"3. [AND] Candidate AND Senator:\n{candidate_and_senator.flatten()}\n")

delivered_speech = model.binaryOp("delivered", target, "speech")
print(f"4. Did '{target}' 'deliver' 'speech'?\n{delivered_speech.flatten()}\n")

print("Story Conclusion: The sparse logic model works and handles larger domains efficiently.")

1. Is 'Sanders' a 'candidate'?
[0.5 0.5]

2. Is 'Sanders' a 'senator'?
[0.5 0.5]

3. [AND] Candidate AND Senator:
[0.25 0.75]

4. Did 'Sanders' 'deliver' 'speech'?
[0.5 0.5]

Story Conclusion: The sparse logic model works and handles larger domains efficiently.
